# Verificação de duplicidade do ID do domicílio

A variável `ID_DOMICILIO` corresponde ao identificador `HHX` da pesquisa NHIS, atribuído ao domicílio. O levantamento começa com o cadastro dos moradores e, em seguida, realiza uma entrevista de adulto selecionado ou, quando aplicável, de criança selecionada. Assim, o identificador continua sendo do domicílio — não um identificador pessoal —, mas, na base pública de adultos usada neste projeto, cada domicílio contribui com o registro de um adulto selecionado.

Por esse desenho, nesta base de adultos, `ID_DOMICILIO` funciona na prática como uma chave única para cada registro de adulto entrevistado: se não houver repetição, haverá uma linha por domicílio e pelo adulto selecionado correspondente. Ainda assim, não deve ser tratado como um ID de pessoa reutilizável fora desta base ou capaz de distinguir diferentes moradores da mesma residência. A verificação abaixo confirmará se o identificador aparece mais de uma vez na base pré-processada.

In [1]:
import pandas as pd
from pathlib import Path

ID_DOMICILIO = "ID_DOMICILIO"
arquivo_base = Path("../dados/dados_limpos") / "adult25_limpo.csv"

if not arquivo_base.is_file():
    raise FileNotFoundError(f"Arquivo de dados limpos não encontrado: {arquivo_base.resolve()}")

df_silencioso = pd.read_csv(arquivo_base, low_memory=False)
if ID_DOMICILIO not in df_silencioso.columns:
    raise KeyError(f"A coluna obrigatória {ID_DOMICILIO!r} não existe na base.")

ids_domicilio = df_silencioso[ID_DOMICILIO]
quantidade_ausente = int(ids_domicilio.isna().sum())
if quantidade_ausente:
    raise ValueError(f"Foram encontrados {quantidade_ausente} valores ausentes em {ID_DOMICILIO}.")

ids_repetidos = ids_domicilio[ids_domicilio.duplicated(keep=False)]
quantidade_ids_repetidos = int(ids_repetidos.nunique())
quantidade_registros_com_id_repetido = int(ids_domicilio.duplicated(keep=False).sum())

print(f"Registros analisados: {len(df_silencioso):,}")
print(f"IDs de domicílio distintos: {ids_domicilio.nunique():,}")
print(f"IDs de domicílio repetidos: {quantidade_ids_repetidos:,}")
print(f"Registros associados a IDs repetidos: {quantidade_registros_com_id_repetido:,}")

if quantidade_ids_repetidos:
    print("\nFrequência dos IDs_DOMICILIO repetidos:")
    display(ids_repetidos.value_counts().rename_axis(ID_DOMICILIO).reset_index(name="quantidade_registros"))
else:
    print("Verificação concluída: ID_DOMICILIO não se repete na base analisada.")

assert ids_domicilio.is_unique, "A verificação encontrou IDs_DOMICILIO repetidos."

Registros analisados: 24,215
IDs de domicílio distintos: 24,215
IDs de domicílio repetidos: 0
Registros associados a IDs repetidos: 0
Verificação concluída: ID_DOMICILIO não se repete na base analisada.


### Interpretação

O arquivo `adult25_limpo.csv` contém 24.215 registros e 24.215 valores distintos de `ID_DOMICILIO`; a verificação encontra zero IDs repetidos. Assim, nesta base de adultos, há uma única linha por domicílio e pelo adulto selecionado para a entrevista.

`ID_DOMICILIO` poderá ser usado para identificar de forma única o registro do adulto selecionado dentro desta base. Contudo, seu significado original permanece domiciliar: não identifica nominalmente a pessoa, não distingue moradores diferentes da mesma casa e não deve ser interpretado como um identificador individual fora deste arquivo.